In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
%%capture
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

In [3]:
%%capture
!pip install python-terrier sentence-transformers

# SE 2026 - Lab 6: Reranking

Today, we will look at reranking!

Reranking is a multi-stage retrieval approach that combines the strengths of efficient lexical retrieval models with more expressive but computationally expensive neural models.

For example, **lexical retrieval models**, like BM25, can efficiently retrieve a large set of candidate documents. These models are highly effective at maximizing recall, but they rely primarily on lexical overlap and may not fully capture semantic relationships between queries and documents.

![Image showcasing the advantages of reranking](https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2F906c3c0f8fe637840f134dbf966839ef89ac7242-3443x1641.png&w=3840&q=75)

**Dense neural models**—such as a BERT-based bi-encoder or cross-encoder  incorporate contextual and semantic information, allowing for more accurate estimation of query–document relevance. However, these models require significantly more computation at inference time, especially when searching across entire corpora. By using them instead as *rerankers*, we can apply them on a smaller subset of $K$ documents obtained by our retriever.  

We combine these two methods into one two-step pipeline which can improve efficiency and effectiveness. In the first step, we take an efficient model that can prioritise recall over large document collections, and reduces our pool of documents to $K$. Our second step steps our pool of $K$ documents and refines the ranking quality of the most promising candidates.

The set-up can be abstracted as the below pipeline. To implement it, we will rely on PyTerrier's pipeline functionality.

```
                            Query
                              │
                              ▼
                
                   (1) RETRIEVAL STAGE
                  BM25 ──► Top K Documents

                              │
                              ▼
                  (2)  RANKING STAGE
                 Embeddings ──► Similarity(q, d)
               
                              │
                              ▼
                      Re-ranked Results
```


In [4]:
import numpy as np
import pandas as pd
import warnings
from pprint import pprint
from pyterrier.measures import *
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

In [5]:
%%capture

# Init PyTerrier
%env JAVA_HOME=/root/.sdkman/candidates/java/current
import pyterrier as pt
if not pt.started():
    # pt.init(boot_packages=["com.github.terrierteam:terrier-prf:-SNAPSHOT"])  # Initialisation package for RM3
    pt.init(boot_packages=[])  # Initialisation package for RM3

## Systems Setup

We will start by building an index of our data collection and a few systems in PyTerrier.

*Note: For this lab, we need to include metadata in our indexer, to ensure that text is fed to our reranking model. Please observed below*

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [38]:
docs = pd.read_csv('drive/MyDrive/data/lab_docs.csv', dtype={'docno': str, 'text': str})
topics = pd.read_csv('drive/MyDrive/data/lab_topics.csv', dtype={'qid': str})
qrels = pd.read_csv('drive/MyDrive/data/lab_qrels.csv', dtype={'qid': str, 'docno': str, 'label': int})

In [111]:
# Build index
indexer = pt.IterDictIndexer("./indexes/pt_index_default", overwrite=True, blocks=True,
                                  meta={ ### NOTE: We need to include this meta information for our reranker
                                "docno": 20,
                                "text": 4096   # or larger depending on docs
                            })
index_ref = indexer.index(docs.to_dict(orient='records'))
index = pt.IndexFactory.of(index_ref)
print(index.getCollectionStatistics().toString())

Number of documents: 2453
Number of terms: 23693
Number of postings: 208487
Number of fields: 0
Number of tokens: 273373
Field names: []
Positions:   true



## Our Retriever

We start by building our retriever. We will use the basic BM25 retriever with the baseline settings (make sure to include text metadata!). In the final assignment, use your best performing retriever.

We will evaluate using $nDCG@k$, $MRR@k$, $Precision@k$ and $Recall@k$. In the final assignment, you will be asked to look at $k\in\{5,10,20\}$. In this notebook, we look at just $k=10$.

We set $K=100$. In the final assignment you can select a $K\in\{100, 500, 1000\}$.

In [102]:

BM25 = pt.terrier.Retriever(index, wmodel="BM25", metadata=['docno','text'])
#Note: we need to pass the text metadata here so we can feed it into our Rerankers!
K = 100

In [103]:
# Evaluate systems on the topics using the PyTerrier Experiment interface
# NDCG, MRR, Precision and Recall at 5, 10 and 20 c
pt.Experiment(
    retr_systems=[BM25],
    names=['BM25'],
    topics=topics,
    qrels=qrels,
    eval_metrics=[R@10, nDCG@10, P@10, MRR@10],
    round=4
)

,name,P@10,R@10,nDCG@10,RR@10
0,BM25,0.7667,0.3869,0.8425,1.0


In [104]:
## To select K documents from BM25, we use %

BM25 % K

(TerrierRetr(BM25) >> RankCutoff(100))

## **Reranking**

We now create our reranking model. While there are many specialised rerankers available online (You can read about some options [here](https://medium.com/@abheshith7/mastering-reranking-in-rag-from-basic-retrieval-to-advanced-methods-db297530361a)), we will rely on Sentence-BERT (i.e. sentence transformers) to implement both a biencoder reranker and crossencoder reranker. For the final assignment, you only need to implement a **biencoder reranker**. *Note: You may need to create an HF_TOKEN and add it to your Colab Secrets to load huggingface models on sentence transformers.*

For reference purposes, we use the same base model for both the biencoder and the crossencoder.

### Biencoder
![Biencoder example](https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2F4509817116ab72e27bae809c38cb48fbf1578b5d-2760x1420.png&w=3840&q=75)
Biencoders take two separate documents (or a document and a query) and independently map them into the same vector space, and then calculate their proximity (i.e. via cosine similarity). However, this means that *all possible document information* is compressed, without any contextual information of the query. This can expedite search time (as we can pre-compute query vectors when loading our database), but we can lose some information in the compression.

Theoretically, *any* model that can generate text embeddings can be used as a biencoder.

### Crossencoder

![Crossencoder example](https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2F9f0d2f75571bb58eecf2520a23d300a5fc5b1e2c-2440x1100.png&w=3840&q=75)
Crossencoders encode both the query and the document *together* to obtain the similarity score.

This means that cross encoders must be specifically trained to output scores. You can see available cross encoders [here](https://huggingface.co/models?library=sentence-transformers&pipeline_tag=text-ranking).


For implementation of rerankers on PyTerrier, we refer to [this PyTerrier notebook](https://colab.research.google.com/github/terrier-org/pyterrier/blob/master/examples/notebooks/sentence_transformers.ipynb#scrollTo=pYM3VCONjhb_).

In [123]:
from sentence_transformers import CrossEncoder, SentenceTransformer
from sentence_transformers.util import cos_sim


bimodel = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2') ## Any HF embedding-model can be put here
crossmodel = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2') ## Requires a crossencoder-trained model!

def _crossencoder_apply(df : pd.DataFrame):
    return crossmodel.predict(list(zip(df['query'].values, df['text'].values)))

def _biencoder_apply(df : pd.DataFrame):
    query_embs = bimodel.encode(df['query'].values, normalize_embeddings=True) # Remember to normalise embeddings
    doc_embs = bimodel.encode(df['text'].values, normalize_embeddings=True)
    scores =  cos_sim(query_embs, doc_embs)
    #return scores[0] ## In original notebook,
    return scores.diagonal().cpu().numpy() # Gives diagonal matching

cross_encT = pt.apply.doc_score(_crossencoder_apply, batch_size=128)
bi_encT = pt.apply.doc_score(_biencoder_apply, batch_size=128)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [116]:
BM25 % K >> bi_encT

(TerrierRetr(BM25) >> RankCutoff(100) >> pt.apply.doc_score())

In [117]:
BM25 % K >> cross_encT

(TerrierRetr(BM25) >> RankCutoff(100) >> pt.apply.doc_score())

In [124]:
pt.Experiment(
    retr_systems=[ BM25, BM25 % K >> bi_encT, BM25 % K >> cross_encT ],
    names=["BM25", "BM25 >> BiEncoder", "BM25 >> CrossEncoder"],
    topics=topics,
    qrels=qrels,
    eval_metrics=[R@10, nDCG@10, P@10, MRR@10],
    round=4
)

,name,P@10,R@10,nDCG@10,RR@10
0,BM25,0.7667,0.3869,0.8425,1.0
1,BM25 >> BiEncoder,0.8333,0.4517,0.8978,1.0
2,BM25 >> CrossEncoder,0.8667,0.4777,0.9141,1.0


If you're interested in more complicated reranking set-ups available with PyTerrier. Check out [this documentation for examples](https://pyterrier.readthedocs.io/en/latest/neural.html)

In [125]:
## We can inspect the top documents for the first query for each system below

res = (BM25 % 10).transform(topics.head(1))
res

,qid,docid,docno,text,rank,score,query
0,1015979,205,1015979,the president is responsible for both the chil...,0,20.927815,president of chile
1,1015979,2435,229754,the war saw a confrontation between the chilea...,1,18.834027,president of chile
2,1015979,2417,1186821,she is of basque descent larra n currently sit...,2,18.584731,president of chile
3,1015979,546,2226456,constructed in 1929 in the spanish colonial re...,3,14.702179,president of chile
4,1015979,549,1514612,at age fifteen he returned to chile completing...,4,13.293413,president of chile
5,1015979,2399,496876,the cream sauce usually has milk double cream ...,5,10.561582,president of chile
6,1015979,1200,1304956,he commanded the 4th ss polizei division and t...,6,10.142057,president of chile
7,1015979,1817,1053174,after lagos obtained his ph d in the u s he an...,7,8.959842,president of chile
8,1015979,1031,787359,where they are called sesos in spanish and are...,8,8.074396,president of chile
9,1015979,1893,1156486,the mexican state of zacatecas is one of the m...,9,8.074396,president of chile


In [126]:
res = (BM25 % K >> cross_encT).transform(topics.head(1))
res.sort_values('rank')

,qid,docid,docno,text,score,query,rank
0,1015979,205,1015979,the president is responsible for both the chil...,8.243772,president of chile,0
1,1015979,2435,229754,the war saw a confrontation between the chilea...,4.308006,president of chile,1
3,1015979,546,2226456,constructed in 1929 in the spanish colonial re...,2.829351,president of chile,2
2,1015979,2417,1186821,she is of basque descent larra n currently sit...,0.554739,president of chile,3
4,1015979,549,1514612,at age fifteen he returned to chile completing...,-0.898898,president of chile,4
...,...,...,...,...,...,...,...
69,1015979,641,1311910,in 1952 it was converted into the largest pass...,-10.890496,president of chile,95
45,1015979,603,1581012,the aircraft were desperately needed to bolste...,-10.901766,president of chile,96
51,1015979,1895,663106,originally published in west germany in 1950 t...,-10.919436,president of chile,97
53,1015979,1226,245276,as the name suggests it was responsible for ai...,-10.990652,president of chile,98


In [127]:
res = (BM25 % K >> bi_encT).transform(topics.head(1))
res.sort_values('rank')

,qid,docid,docno,text,score,query,rank
0,1015979,205,1015979,the president is responsible for both the chil...,0.725874,president of chile,0
4,1015979,549,1514612,at age fifteen he returned to chile completing...,0.636872,president of chile,1
1,1015979,2435,229754,the war saw a confrontation between the chilea...,0.612475,president of chile,2
3,1015979,546,2226456,constructed in 1929 in the spanish colonial re...,0.552363,president of chile,3
30,1015979,808,1119171,he was of basque descent and a member of the b...,0.492166,president of chile,4
...,...,...,...,...,...,...,...
96,1015979,2262,1365636,in december 1920 the texas national guard was ...,-0.007304,president of chile,95
94,1015979,508,1454876,the topic of using germans as forced labour fo...,-0.010339,president of chile,96
89,1015979,2201,1016767,wells studied art in london under john james b...,-0.027305,president of chile,97
54,1015979,1545,915625,with a lower drag coefficient than most contem...,-0.033835,president of chile,98
